In [1]:
import pandas as pd
import numpy as np

In [4]:
import json
import joblib
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

data_path = Path("../data/interim/listings_parsed_v7.csv")
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(data_path, dtype={"jnc": str})
assert len(df) == 183490, f"Expected 183490 rows, got {len(df)}"

FEATURES = [
    "area_m2", "age_years", "floor_min", "stories_above",
    "walk_min_nearest", "n_rooms", "has_L", "has_D", "has_K",
    "has_S", "is_1R",
]
CAT = ["municipality"]

PARAMS = {
    "colsample_bytree": 0.6978,
    "learning_rate": 0.0407,
    "max_depth": 9,
    "min_child_weight": 6,
    "n_estimators": 860,
    "reg_lambda": 1.1218,
    "subsample": 0.6786,
}


def make_pipe(**extra):
    return Pipeline([
        ("pre", ColumnTransformer([
            ("num", "passthrough", FEATURES),
            ("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
        ])),
        ("reg", XGBRegressor(
            tree_method="hist",
            device="cuda",
            random_state=0,
            n_jobs=-1,
            **PARAMS,
            **extra,
        )),
    ])


X = df[FEATURES + CAT]
y = df["Price_yen"]

mean_model = make_pipe()
mean_model.fit(X, y)
joblib.dump(mean_model, models_dir / "xgb_mean.joblib")

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
fit_idx, cal_idx = next(splitter.split(df, groups=df["group_coarse"]))

z = np.log1p(df["Price_yen"].to_numpy())
quantile_predictions = {}

for alpha, filename in [
    (0.1, "xgb_q10.joblib"),
    (0.5, "xgb_q50.joblib"),
    (0.9, "xgb_q90.joblib"),
]:
    model = make_pipe(
        objective="reg:quantileerror",
        quantile_alpha=alpha,
    )
    model.fit(X.iloc[fit_idx], z[fit_idx])
    joblib.dump(model, models_dir / filename)
    quantile_predictions[alpha] = model.predict(X.iloc[cal_idx])

lo = quantile_predictions[0.1]
mid = quantile_predictions[0.5]
hi = quantile_predictions[0.9]

lo = np.minimum(lo, mid)
hi = np.maximum(hi, mid)
z_cal = z[cal_idx]
score = np.maximum(lo - z_cal, z_cal - hi)

edges = np.array([0.0, 80_000.0, 150_000.0, np.inf])
labels = ["<80k", "80-150k", ">150k"]
bands = pd.cut(
    np.expm1(mid),
    bins=edges,
    labels=labels,
    right=False,
    include_lowest=True,
)

n = len(score)
quantile_level = min(1.0, np.ceil((n + 1) * 0.8) / n)
q_global = float(np.quantile(score, quantile_level, method="higher"))

q_band = {}
for label in labels:
    values = score[np.asarray(bands == label)]
    q_band[label] = (
        float(np.quantile(values, quantile_level, method="higher"))
        if len(values) >= 200
        else q_global
    )

with open(models_dir / "calibration.json", "w", encoding="utf-8") as f:
    json.dump({
        "q_band": q_band,
        "q_global": q_global,
        "edges": [0.0, 80000.0, 150000.0, None],
        "features": FEATURES,
        "cat": CAT,
        "params": PARAMS,
    }, f, indent=2)


mean_mae = float(np.mean(np.abs(mean_model.predict(X) - y)))
calibration_q = np.array([
    q_band[label] for label in bands.astype(str)
])
coverage = float(np.mean(score <= calibration_q))

print(f"Mean model in-sample MAE: {mean_mae:,.0f} yen")
print(f"Calibrated calibration coverage: {coverage:.3f}")
print("Calibration q_band:", q_band)

examples = pd.DataFrame([
    {
        "area_m2": 25, "age_years": 10, "floor_min": 1,
        "stories_above": 2, "walk_min_nearest": 5, "n_rooms": 1,
        "has_L": 0, "has_D": 0, "has_K": 1, "has_S": 0,
        "is_1R": 0, "municipality": "世田谷区",
    },
    {
        "area_m2": 60, "age_years": 10, "floor_min": 1,
        "stories_above": 5, "walk_min_nearest": 10, "n_rooms": 2,
        "has_L": 1, "has_D": 1, "has_K": 1, "has_S": 0,
        "is_1R": 0, "municipality": "八王子市",
    },
    {
        "area_m2": 15, "age_years": 10, "floor_min": 1,
        "stories_above": 2, "walk_min_nearest": 5, "n_rooms": 1,
        "has_L": 0, "has_D": 0, "has_K": 0, "has_S": 0,
        "is_1R": 1, "municipality": "足立区",
    },
])

example_mid = np.expm1(
    joblib.load(models_dir / "xgb_q50.joblib").predict(examples)
)
example_lo = np.expm1(
    joblib.load(models_dir / "xgb_q10.joblib").predict(examples)
)
example_hi = np.expm1(
    joblib.load(models_dir / "xgb_q90.joblib").predict(examples)
)

example_lo = np.minimum(example_lo, example_mid)
example_hi = np.maximum(example_hi, example_mid)

example_bands = pd.cut(
    example_mid,
    bins=edges,
    labels=labels,
    right=False,
    include_lowest=True,
)

raw = {a: joblib.load(models_dir / f"xgb_q{int(a*100)}.joblib").predict(examples)
       for a in (0.1, 0.5, 0.9)}
lo_l = np.minimum(raw[0.1], raw[0.5])
mid_l = raw[0.5]
hi_l = np.maximum(raw[0.9], raw[0.5])

mid_y = np.expm1(mid_l)
ex_bands = pd.cut(mid_y, bins=edges, labels=labels, right=False, include_lowest=True)
qs = np.array([q_band[str(b)] for b in ex_bands])

lo_y = np.expm1(lo_l - qs)
hi_y = np.expm1(hi_l + qs)

for i, row in examples.iterrows():
    print(f"{row['municipality']}: estimate={mid_y[i]:,.0f}, "
          f"80% interval=({lo_y[i]:,.0f}, {hi_y[i]:,.0f})")

Mean model in-sample MAE: 7,472 yen
Calibrated calibration coverage: 0.800
Calibration q_band: {'<80k': 0.019587062233691555, '80-150k': 0.021977713028080714, '>150k': 0.053374457214120596}
世田谷区: estimate=82,131, 80% interval=(69,683, 96,478)
八王子市: estimate=105,992, 80% interval=(88,960, 138,192)
足立区: estimate=54,264, 80% interval=(49,389, 65,064)
